# 1. Overall SaaS KPIs

This section establishes the overall business baseline by calculating key SaaS performance indicators.

Key metrics:
- Total customers
- Total subscriptions
- Active subscriptions
- Total MRR
- Total ARR
- Average MRR per subscription

In [2]:
SELECT
    COUNT(DISTINCT account_id) AS total_customers,
    COUNT(DISTINCT subscription_id) AS total_subscriptions,
    COUNT(DISTINCT CASE 
        WHEN end_date IS NULL THEN subscription_id 
    END) AS active_subscriptions,
    SUM(CASE 
        WHEN end_date IS NULL THEN mrr_amount 
        ELSE 0 
    END) AS total_mrr,
    SUM(CASE 
        WHEN end_date IS NULL THEN arr_amount 
        ELSE 0 
    END) AS total_arr,
    ROUND(AVG(CASE 
        WHEN end_date IS NULL THEN mrr_amount 
    END), 2) AS avg_mrr_per_active_subscription
FROM subscriptions;

StatementMeta(, 410ddc12-1bd0-475c-9090-4e00317b9a7d, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 6 fields>

## Result

The SaaS dataset contains 500 customers and 5,000 subscription records.

- 4,514 subscriptions are currently active.
- Total active MRR is 10,159,608.
- Total active ARR is 12,191,529.60.
- Average MRR per active subscription is 2,250.69.

These metrics establish the overall revenue and subscription baseline for the subsequent business analysis.

# 2. Revenue & MRR/ARR Analysis

This section examines recurring revenue performance over time.

Key metrics:
- Monthly Recurring Revenue (MRR)
- Annual Recurring Revenue (ARR)
- Month-over-month MRR growth
- Revenue contribution by subscription plan

In [3]:
SELECT
    DATE_FORMAT(start_date, 'yyyy-MM') AS month,
    SUM(mrr_amount) AS total_mrr,
    SUM(arr_amount) AS total_arr
FROM subscriptions
GROUP BY DATE_FORMAT(start_date, 'yyyy-MM')
ORDER BY month;

StatementMeta(, 410ddc12-1bd0-475c-9090-4e00317b9a7d, 4, Finished, Available, Finished, False)

<Spark SQL result set with 24 rows and 3 fields>

## Result

The monthly revenue analysis shows a strong upward trend in recurring revenue from January 2023 to December 2024.

- MRR increased from 4,684 in January 2023 to 2,273,427 in December 2024.
- ARR increased from 56,208 in January 2023 to 27,281,124 in December 2024.
- Revenue growth was generally positive across the analysis period, with some month-to-month fluctuations.
- The highest MRR and ARR in the dataset were recorded in December 2024.

This indicates substantial growth in the SaaS subscription base and recurring revenue over the two-year period.

### 2.1 MRR Contribution by Plan

In [4]:
SELECT
    plan_tier,
    COUNT(DISTINCT subscription_id) AS total_subscriptions,
    SUM(mrr_amount) AS total_mrr,
    SUM(arr_amount) AS total_arr,
    ROUND(AVG(mrr_amount), 2) AS avg_mrr
FROM subscriptions
GROUP BY plan_tier
ORDER BY total_mrr DESC;

StatementMeta(, 410ddc12-1bd0-475c-9090-4e00317b9a7d, 5, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 5 fields>

## Result

Enterprise is the strongest contributor to recurring revenue, followed by Pro and Basic plans.

- **Enterprise:** 1,723 subscriptions, MRR of 8,473,221, and the highest average MRR of 4,917.71.
- **Pro:** 1,675 subscriptions, MRR of 2,105,089, with an average MRR of 1,256.77.
- **Basic:** 1,602 subscriptions, MRR of 760,437, with an average MRR of 474.68.

Enterprise generates substantially more recurring revenue per subscription than the other plans, indicating that higher-tier customers are the major revenue contributors.

### 2.2 Month-over-Month MRR Growth

This analysis measures how monthly recurring revenue changes over time and identifies periods of acceleration or decline.

In [5]:
WITH monthly_mrr AS (
    SELECT
        DATE_FORMAT(start_date, 'yyyy-MM') AS month,
        SUM(mrr_amount) AS total_mrr
    FROM subscriptions
    GROUP BY DATE_FORMAT(start_date, 'yyyy-MM')
),

mrr_with_previous AS (
    SELECT
        month,
        total_mrr,
        LAG(total_mrr) OVER (ORDER BY month) AS previous_mrr
    FROM monthly_mrr
)

SELECT
    month,
    total_mrr,
    previous_mrr,
    ROUND(
        ((total_mrr - previous_mrr) / previous_mrr) * 100,
        2
    ) AS mom_mrr_growth_pct
FROM mrr_with_previous
ORDER BY month;

StatementMeta(, 410ddc12-1bd0-475c-9090-4e00317b9a7d, 6, Finished, Available, Finished, False)

<Spark SQL result set with 24 rows and 4 fields>

### Result

- MRR shows a strong overall upward trend from January 2023 through December 2024.
- The highest month-over-month growth occurred in **February 2023 (136.53%)**.
- MRR declined in **June 2023 (-12.72%)**, **September 2023 (-29.83%)**, **June 2024 (-14.99%)**, and **August 2024 (-8.39%)**.
- The largest decline was observed in **September 2023 (-29.83%)**.
- Growth accelerated again toward the end of 2024, reaching **46.76% in December 2024**.
- Overall, the company demonstrates strong MRR growth despite periodic monthly declines.

## 2.3 MRR by Plan Tier

This section compares subscription performance across Basic, Pro, and Enterprise plans.

Key metrics:
- Total subscriptions
- Total MRR
- Total ARR
- Average MRR per subscription

In [6]:
SELECT
    plan_tier,
    COUNT(DISTINCT subscription_id) AS total_subscriptions,
    SUM(mrr_amount) AS total_mrr,
    SUM(arr_amount) AS total_arr,
    ROUND(AVG(mrr_amount), 2) AS avg_mrr
FROM subscriptions
GROUP BY plan_tier
ORDER BY total_mrr DESC;

StatementMeta(, 410ddc12-1bd0-475c-9090-4e00317b9a7d, 7, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 5 fields>

### Result

- **Enterprise** is the strongest revenue-generating plan, contributing **8,473,221 MRR** from 1,723 subscriptions.
- **Pro** contributes **2,105,089 MRR** from 1,675 subscriptions.
- **Basic** contributes **760,437 MRR** from 1,602 subscriptions.
- Enterprise has the highest average MRR per subscription at **4,917.71**, compared with **1,256.77** for Pro and **474.68** for Basic.
- Although subscription counts are relatively similar across the three plans, Enterprise generates substantially more recurring revenue because of its much higher average subscription value.
- This indicates that **Enterprise customers are the primary driver of SaaS recurring revenue**.

## 2.4 Subscription Status & Churn

This section evaluates the current subscription status and overall customer churn.

Key metrics:
- Total subscriptions
- Active subscriptions
- Churned subscriptions
- Overall churn rate
- Active subscription rate

The analysis helps assess the health of the subscription base and identify the overall level of customer attrition.

In [7]:
SELECT
    COUNT(DISTINCT subscription_id) AS total_subscriptions,
    SUM(CASE WHEN churn_flag = TRUE THEN 1 ELSE 0 END) AS churned_subscriptions,
    SUM(CASE WHEN churn_flag = FALSE THEN 1 ELSE 0 END) AS active_subscriptions,
    ROUND(
        100.0 * SUM(CASE WHEN churn_flag = TRUE THEN 1 ELSE 0 END)
        / COUNT(DISTINCT subscription_id),
        2
    ) AS churn_rate_pct,
    ROUND(
        100.0 * SUM(CASE WHEN churn_flag = FALSE THEN 1 ELSE 0 END)
        / COUNT(DISTINCT subscription_id),
        2
    ) AS active_rate_pct
FROM subscriptions;

StatementMeta(, 410ddc12-1bd0-475c-9090-4e00317b9a7d, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 5 fields>

### Result

- The dataset contains **5,000 total subscriptions**.
- **4,514 subscriptions are active**, representing an **active rate of 90.28%**.
- **486 subscriptions have churned**, resulting in an overall **churn rate of 9.72%**.
- The majority of subscriptions remain active, indicating a relatively healthy subscription base.
- However, nearly **1 in 10 subscriptions has churned**, making churn an important area for further analysis.

## 2.5 Churn by Plan Tier

This section compares customer churn across subscription plan tiers.

Key metrics:
- Total subscriptions by plan
- Churned subscriptions by plan
- Churn rate by plan

The analysis helps identify which pricing tier has the highest customer attrition and may require retention-focused attention.

In [8]:
SELECT
    plan_tier,
    COUNT(DISTINCT subscription_id) AS total_subscriptions,
    SUM(CASE WHEN churn_flag = TRUE THEN 1 ELSE 0 END) AS churned_subscriptions,
    ROUND(
        100.0 * SUM(CASE WHEN churn_flag = TRUE THEN 1 ELSE 0 END)
        / COUNT(DISTINCT subscription_id),
        2
    ) AS churn_rate_pct
FROM subscriptions
GROUP BY plan_tier
ORDER BY churn_rate_pct DESC;

StatementMeta(, 410ddc12-1bd0-475c-9090-4e00317b9a7d, 9, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 4 fields>

### Result

- **Enterprise** has the highest churn rate at **9.98%**, with **172 churned subscriptions** out of 1,723.
- **Pro** has a churn rate of **9.67%**, with **162 churned subscriptions** out of 1,675.
- **Basic** has the lowest churn rate at **9.49%**, with **152 churned subscriptions** out of 1,602.
- Churn rates are relatively close across all three plans, ranging from **9.49% to 9.98%**.
- Enterprise has the highest churn rate, but the difference between plans is small, so plan tier alone does not appear to be a major differentiator of churn in this dataset.

## 2.6 Churn by Industry

This section analyzes subscription churn across different customer industries.

Key metrics:
- Total customers by industry
- Churned customers by industry
- Churn rate by industry

The analysis helps identify industries with relatively higher customer attrition and potential retention concerns.

In [11]:
SELECT
    a.industry,
    COUNT(DISTINCT s.account_id) AS total_customers,
    COUNT(DISTINCT CASE
        WHEN s.churn_flag = TRUE THEN s.account_id
    END) AS churned_customers,
    ROUND(
        100.0 * COUNT(DISTINCT CASE
            WHEN s.churn_flag = TRUE THEN s.account_id
        END)
        / COUNT(DISTINCT s.account_id),
        2
    ) AS churn_rate_pct
FROM subscriptions s
JOIN accounts a
    ON s.account_id = a.account_id
GROUP BY a.industry
ORDER BY churn_rate_pct DESC;

StatementMeta(, 410ddc12-1bd0-475c-9090-4e00317b9a7d, 12, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 4 fields>

### Result

- **HealthTech** has the highest churn rate at **70.83%**, with 68 churned customers out of 96.
- **DevTools** follows closely at **70.80%**, with 80 churned customers out of 113.
- **Cybersecurity** has a churn rate of **65.00%**.
- **EdTech** has a churn rate of **55.70%**.
- **FinTech** has the lowest churn rate at **49.11%**.

**Insight:** HealthTech and DevTools show the highest customer churn, indicating that these industries may require stronger retention strategies.

## 2.7 Churn by Referral Source

This section analyzes customer churn across different referral sources.

Key metrics:
- Total customers by referral source
- Churned customers by referral source
- Churn rate by referral source

The analysis helps identify acquisition channels associated with higher or lower customer retention.

In [12]:
SELECT
    a.referral_source,
    COUNT(DISTINCT s.account_id) AS total_customers,
    COUNT(DISTINCT CASE
        WHEN s.churn_flag = TRUE THEN s.account_id
    END) AS churned_customers,
    ROUND(
        100.0 * COUNT(DISTINCT CASE
            WHEN s.churn_flag = TRUE THEN s.account_id
        END)
        / COUNT(DISTINCT s.account_id),
        2
    ) AS churn_rate_pct
FROM subscriptions s
JOIN accounts a
    ON s.account_id = a.account_id
GROUP BY a.referral_source
ORDER BY churn_rate_pct DESC;

StatementMeta(, 410ddc12-1bd0-475c-9090-4e00317b9a7d, 13, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 4 fields>

### Result

- **Organic** customers have the highest churn rate at **68.42%**.
- **Other** referral sources have a churn rate of **63.11%**.
- **Ads** have a churn rate of **62.24%**.
- **Event** referrals have a churn rate of **60.42%**.
- **Partner** referrals have the lowest churn rate at **56.18%**.

**Insight:** Customers acquired through organic sources show the highest churn, while partner-referred customers show the strongest retention among the referral sources analyzed.

## 2.8 Customer Subscription Duration

This section analyzes the duration of customer subscriptions to understand retention and customer lifetime behavior.

Key metrics:
- Average subscription duration
- Minimum and maximum subscription duration
- Average duration of churned subscriptions
- Average duration of active subscriptions

**Business objective:** Determine whether customers who churn tend to have shorter subscription lifetimes than customers who remain active.

In [1]:
SELECT
    ROUND(AVG(DATEDIFF(
        COALESCE(end_date, CURRENT_DATE()),
        start_date
    )), 2) AS avg_subscription_days,

    MIN(DATEDIFF(
        COALESCE(end_date, CURRENT_DATE()),
        start_date
    )) AS min_subscription_days,

    MAX(DATEDIFF(
        COALESCE(end_date, CURRENT_DATE()),
        start_date
    )) AS max_subscription_days,

    ROUND(AVG(CASE
        WHEN churn_flag = TRUE THEN DATEDIFF(end_date, start_date)
    END), 2) AS avg_churned_subscription_days,

    ROUND(AVG(CASE
        WHEN churn_flag = FALSE THEN DATEDIFF(
            COALESCE(end_date, CURRENT_DATE()),
            start_date
        )
    END), 2) AS avg_active_subscription_days

FROM subscriptions;

StatementMeta(, 219e6d30-7063-4fa1-bdef-0f0989b8dcd8, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 5 fields>

### Result

- The **average subscription duration** is **723.37 days**.
- The minimum subscription duration is **0 days**, indicating some subscriptions started and ended on the same day.
- The maximum subscription duration is **1,345 days**.
- Customers who churned had an average subscription duration of **88.06 days**.
- Active subscriptions have an average duration of **791.77 days**.

**Insight:** Churned customers have a substantially shorter average subscription duration than active customers. This suggests that early customer retention is an important area for the business to focus on.

## 2.9 Revenue at Risk from Churn

This section quantifies the recurring revenue associated with churned subscriptions.

Key metrics:
- Churned subscriptions
- MRR lost from churn
- ARR lost from churn
- Revenue at risk by plan

**Business objective:** Identify which subscription plans contribute the greatest recurring revenue loss from churn and prioritize retention efforts accordingly.

In [2]:
SELECT
    plan_tier,
    COUNT(DISTINCT subscription_id) AS churned_subscriptions,
    SUM(mrr_amount) AS churned_mrr,
    SUM(arr_amount) AS churned_arr,
    ROUND(
        100.0 * SUM(mrr_amount)
        / SUM(SUM(mrr_amount)) OVER (),
        2
    ) AS pct_of_churned_mrr
FROM subscriptions
WHERE churn_flag = TRUE
GROUP BY plan_tier
ORDER BY churned_mrr DESC;

StatementMeta(, 219e6d30-7063-4fa1-bdef-0f0989b8dcd8, 3, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 5 fields>

### Result

- **Enterprise** subscriptions account for the largest share of churned recurring revenue.
- Enterprise has **172 churned subscriptions**, generating **926,345 in churned MRR** and **11,116,140 in churned ARR**.
- Enterprise contributes **78.56% of total churned MRR**.
- Pro has **162 churned subscriptions**, with **180,271 churned MRR** and **2,163,252 churned ARR**.
- Basic has **152 churned subscriptions**, with **72,523 churned MRR** and **870,276 churned ARR**.

**Insight:** Although the number of churned subscriptions is relatively similar across plans, Enterprise customers represent the majority of recurring revenue at risk. Retention efforts should therefore prioritize Enterprise customers because their churn has the greatest financial impact.

## 2.10 Customer Acquisition & New Customers

This section analyzes customer acquisition over time to understand how the SaaS customer base has grown.

Key metrics:
- New customers acquired each month
- Cumulative customer growth
- Customer acquisition by referral source

**Business objective:** Identify customer growth trends and understand which acquisition channels contribute to new customer acquisition.

In [3]:
SELECT
    DATE_FORMAT(signup_date, 'yyyy-MM') AS signup_month,
    COUNT(DISTINCT account_id) AS new_customers,
    SUM(COUNT(DISTINCT account_id)) OVER (
        ORDER BY DATE_FORMAT(signup_date, 'yyyy-MM')
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_customers
FROM accounts
GROUP BY DATE_FORMAT(signup_date, 'yyyy-MM')
ORDER BY signup_month;

StatementMeta(, 219e6d30-7063-4fa1-bdef-0f0989b8dcd8, 4, Finished, Available, Finished, False)

<Spark SQL result set with 24 rows and 3 fields>

### Result

- The customer base grew from **17 customers in January 2023** to **500 customers by December 2024**.
- The highest monthly acquisition was **32 new customers in November 2024**.
- Other strong acquisition months included **31 customers in October 2024**, **27 in March 2024**, and **26 in May 2023 and July 2024**.
- The cumulative customer base increased continuously throughout the available period.

**Insight:** The SaaS business shows consistent customer-base growth, reaching 500 accounts by December 2024. Customer acquisition was strongest during late 2024, particularly in October and November.

## 2.11 Customer Retention by Signup Year

This section analyzes customer retention based on the year in which customers signed up.

Key metrics:
- Total customers by signup year
- Churned customers
- Active customers
- Churn rate
- Retention rate

**Business objective:** Compare retention performance across different customer signup cohorts and identify whether customer retention is improving or declining over time.

In [5]:
WITH customer_status AS (
    SELECT
        a.account_id,
        YEAR(a.signup_date) AS signup_year,
        MAX(CASE WHEN s.churn_flag = TRUE THEN 1 ELSE 0 END) AS has_churned
    FROM accounts a
    JOIN subscriptions s
        ON a.account_id = s.account_id
    GROUP BY
        a.account_id,
        YEAR(a.signup_date)
)

SELECT
    signup_year,
    COUNT(*) AS total_customers,

    SUM(has_churned) AS churned_customers,

    SUM(CASE
        WHEN has_churned = 0 THEN 1
        ELSE 0
    END) AS retained_customers,

    ROUND(
        100.0 * SUM(has_churned) / COUNT(*),
        2
    ) AS churn_rate_pct,

    ROUND(
        100.0 * SUM(CASE
            WHEN has_churned = 0 THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS retention_rate_pct

FROM customer_status
GROUP BY signup_year
ORDER BY signup_year;

StatementMeta(, 219e6d30-7063-4fa1-bdef-0f0989b8dcd8, 6, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 6 fields>

### Key Findings

- **2023 cohort:** 227 customers, with a **62.11% churn rate** and **37.89% retention rate**.
- **2024 cohort:** 273 customers, with a **62.64% churn rate** and **37.36% retention rate**.
- Churn remained consistently high across both signup cohorts.
- The 2024 cohort had a slightly higher churn rate and slightly lower retention rate than the 2023 cohort.


## 2.12 Churn by Billing Frequency

This analysis compares customer churn across different billing frequencies to determine whether billing preference is associated with customer retention.

The analysis calculates:
- Total subscriptions
- Churned subscriptions
- Churn rate

### Business Question

**Which billing frequency has the highest customer churn rate?**

In [6]:
SELECT
    billing_frequency,
    COUNT(DISTINCT subscription_id) AS total_subscriptions,

    SUM(
        CASE
            WHEN churn_flag = TRUE THEN 1
            ELSE 0
        END
    ) AS churned_subscriptions,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN churn_flag = TRUE THEN 1
                ELSE 0
            END
        ) / COUNT(DISTINCT subscription_id),
        2
    ) AS churn_rate_pct

FROM subscriptions

GROUP BY billing_frequency

ORDER BY churn_rate_pct DESC;

StatementMeta(, 219e6d30-7063-4fa1-bdef-0f0989b8dcd8, 7, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 4 fields>

### 2.12 Result

- **Annual subscriptions** have a churn rate of **10.04%** across 2,461 subscriptions.
- **Monthly subscriptions** have a churn rate of **9.41%** across 2,539 subscriptions.
- Annual subscriptions show a slightly higher churn rate than monthly subscriptions.
- The difference is relatively small, suggesting that billing frequency alone may not be a major driver of churn.
